# Check ticket A1: is `coco_candidates.csv` correct?

Run each cell in order. Every check prints **OK** or a message saying what is wrong.

**Step 0.** Upload `data/coco_candidates.csv` from the repo when the next cell asks.

In [ ]:
from google.colab import files
up = files.upload()
csv_name = list(up.keys())[0]
print('uploaded', csv_name)

## Check 1. Shape
2,833 rows, no image twice, every box at least 20% of the image.

In [ ]:
import pandas as pd
df = pd.read_csv(csv_name)
print('rows:', len(df))
print('duplicate image ids:', df['image_id'].duplicated().sum())
print('smallest box fraction:', df['box_area_fraction'].min())
print('OK' if len(df) >= 400 and df['image_id'].duplicated().sum() == 0 and df['box_area_fraction'].min() >= 0.20 else 'PROBLEM: see the numbers above')

## Check 2. Maths
Recompute the fraction from the box and image size in each row. They must match.

In [ ]:
recomputed = (df['box_w'] * df['box_h']) / (df['width'] * df['height'])
diff = (recomputed - df['box_area_fraction']).abs()
print('largest difference:', diff.max())
print('OK' if diff.max() < 0.001 else 'PROBLEM: some rows do not add up')

## Check 3. Independent recount
Download the COCO annotations here on Colab and redo the selection with different code. The set of image ids must be the same as in the CSV.

In [ ]:
!wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip
!unzip -o -q annotations_trainval2017.zip annotations/instances_val2017.json
print('downloaded')

In [ ]:
import json
coco = json.load(open('annotations/instances_val2017.json'))
size = {im['id']: im['width'] * im['height'] for im in coco['images']}

biggest = {}
for a in coco['annotations']:
    if a.get('iscrowd', 0):
        continue
    area = a['bbox'][2] * a['bbox'][3]
    if area > biggest.get(a['image_id'], 0):
        biggest[a['image_id']] = area

kept = {i for i, area in biggest.items() if area / size[i] >= 0.20}
mine = set(df['image_id'])
print('recount kept:', len(kept), ' csv has:', len(mine))
print('in csv but not in recount:', len(mine - kept))
print('in recount but not in csv:', len(kept - mine))
print('OK' if kept == mine else 'PROBLEM: the two selections differ')

## Check 4. Eyes
Twelve random photos with the biggest-object box drawn in red. The box should sit on the thing named in the label.

In [ ]:
import io, random, urllib.request
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

random.seed(0)
sample = df.sample(12, random_state=0)
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for ax, (_, r) in zip(axes.flat, sample.iterrows()):
    url = 'http://images.cocodataset.org/val2017/' + r['file_name']
    im = Image.open(io.BytesIO(urllib.request.urlopen(url).read())).convert('RGB')
    d = ImageDraw.Draw(im)
    d.rectangle([r['box_x'], r['box_y'], r['box_x'] + r['box_w'], r['box_y'] + r['box_h']], outline='red', width=4)
    ax.imshow(im)
    ax.set_title(f"{r['category']} ({r['box_area_fraction']:.0%})")
    ax.axis('off')
plt.tight_layout()
plt.show()

## Extra. How many per group
Useful for ticket A2. `person` and `dining table` will be dropped there.

In [ ]:
print(df['supercategory'].value_counts())
print()
print(df['category'].value_counts().head(15))